In [1]:
from pathlib import Path

import shutil
import pandas as pd
from tqdm import tqdm


In [2]:
ROOT = Path("/Users/ramazanpolat/Desktop/datasets/abdomen/abdomen_project")


# çıktılar
DATA_ROOT = ROOT.parent                             # abdomen/
RAW_TRAIN_DIR = DATA_ROOT / "Eğitim Verisi"     # açılmış dizin
RAW_TEST_DIR = DATA_ROOT / "Yarışma Veri Seti"
BILGI_XLSX = DATA_ROOT / "Bilgi.xlsx"

# çıktılar
OUT_DIR = ROOT / "outputs"
SPLIT_DIR = OUT_DIR / "splits"
CLS_DATA_DIR = OUT_DIR / "cls_data"                 # PNG / NPZ görüntüleri
DET_DATA_DIR = OUT_DIR / "det_data"                 # YOLO formatı
SEG_DATA_DIR = OUT_DIR / "seg_data"                 # nnU-Net formatı
CKPT_DIR = OUT_DIR / "checkpoints"
LOG_DIR = OUT_DIR / "logs"

In [3]:
fold = 0

In [4]:
out_root = Path(OUT_DIR)
fold_dir = out_root / f"fold{fold}"
for sub in ("images/train", "images/val", "labels/train", "labels/val"):
    p = fold_dir / sub
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

In [5]:

manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
train_cases = set(pd.read_csv(SPLIT_DIR / f"fold{fold}_train.csv")["Case Number"])
val_cases = set(pd.read_csv(SPLIT_DIR / f"fold{fold}_val.csv")["Case Number"])

In [6]:
manifest.head()

,case,image_id,source,dicom_path,super_labels,bboxes,anatomical_boundary
0,20001,100007,train,/Users/ramazanpolat/Desktop/datasets/abdomen/E...,1,"1,251,290,262,302",NaN
1,20001,100008,train,/Users/ramazanpolat/Desktop/datasets/abdomen/E...,1,"1,251,291,261,301",NaN
2,20001,100010,train,/Users/ramazanpolat/Desktop/datasets/abdomen/E...,NaN,NaN,Kidney-Bladder
3,20001,100014,comp,/Users/ramazanpolat/Desktop/datasets/abdomen/Y...,NaN,NaN,Colon
4,20001,100017,comp,/Users/ramazanpolat/Desktop/datasets/abdomen/Y...,NaN,NaN,Abdominal Aorta


In [7]:
include_val_negatives = True

In [8]:



# BBox içeren kesitleri pozitif; içermeyenleri (val setinde) background olarak yaz
from abdomen_project.src.detection import _write_dataset_yaml, _write_slice_png, _write_yolo_label


for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc=f"YOLO fold{fold}"):
    case = int(row["case"])
    if case in train_cases:
        split = "train"
    elif case in val_cases:
        split = "val"
    else:
        continue
    bboxes_raw = str(row.get("bboxes") or "")
    has_bbox = len(bboxes_raw) > 0
    if split == "train" and not has_bbox:
        # Eğitimde bbox'ı olmayan kesitleri ekleme (sinyal daha temiz)
        continue
    if not include_val_negatives and split == "val" and not has_bbox:
        continue
    try:
        img_path, h, w = _write_slice_png(row, fold_dir / "images" / split)
    except Exception as exc:
        print(f"[skip] {row['dicom_path']}: {exc}")
        continue
    label_path = fold_dir / "labels" / split / (img_path.stem + ".txt")
    _write_yolo_label(bboxes_raw, h, w, label_path)
_write_dataset_yaml(fold_dir)

/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
YOLO fold0:   0%|          | 0/39268 [00:00<?, ?it/s]/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/lib/python3.9/site-packages/pydicom/pixel_data_handlers/numpy_handler.py:250: UserWarning: The length of the pixel data in the dataset (1048576 bytes) indicates it contains excess padding. 524288 bytes will be removed from the end of the data
  warnings.warn(msg)
YOLO fold0:   0%|          | 2/39268 [00:00<22:08, 29.55it/s]


ValueError: invalid literal for int() with base 10: 'nan'